[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/03_ONNX_Architecture_and_Internals/02_Nodes_Edges_and_Tensors/Nodes_Edges_and_Tensors_Deep_Dive.ipynb)

# 3.2 Nodes, Edges, and Tensors — Deep Dive

Zoom into ONNX's wire format: **NodeProto** (operator invocations), **implicit edges** (data flow via SSA naming), and **tensor element types** (the foundation of ONNX's type system).

---

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [NodeProto — Anatomy of a Node](#section-1) | Fields, schemas, and semantics |
| 2 | [Operator Schemas](#section-2) | Type constraints and signatures |
| 3 | [Edges as Data Flow](#section-3) | Implicit wiring and fan-out |
| 4 | [Tensor Types and Element Types](#section-4) | The complete ONNX type catalog |
| 5 | [Tensor Algebra in ONNX](#section-5) | Shape rules for common operators |
| 6 | [Attributes vs Inputs](#section-6) | The static/dynamic boundary |
| 7 | [Building and Inspecting Nodes](#section-7) | Code walkthrough |
| 8 | [Key Takeaways & Interview Questions](#section-8) | Summary |

### Prerequisites

- Completed **3.1 Computation Graph Basics**
- Understanding of tensor shapes and broadcasting

<a id='section-1'></a>
## Section 1: NodeProto — Anatomy of a Node

### Definition

A `NodeProto` represents a **single operator invocation** in the computation graph:

$$v = (\text{op\_type}, [t_1, \ldots, t_k], [t'_1, \ldots, t'_l], \{a_1: c_1, \ldots\})$$

### NodeProto Fields

```
NodeProto
├── name       : string         ← optional human-readable identifier
├── op_type    : string         ← operator name ("MatMul", "Conv", "Relu")
├── domain     : string         ← operator domain ("" = standard ONNX)
├── input      : repeated string ← ordered tensor names consumed
├── output     : repeated string ← ordered tensor names produced
├── attribute  : repeated AttributeProto ← static operator parameters
├── doc_string : string         ← optional documentation
└── overload   : string         ← function overload identifier (rare)
```

### Field Semantics

| Field | Type | Mutable? | Description |
|-------|------|:--------:|-------------|
| `op_type` | `string` | No | Which operator (must exist in opset) |
| `domain` | `string` | No | Which domain (empty = standard) |
| `input` | `[string]` | No | Ordered input tensor names |
| `output` | `[string]` | No | Ordered output tensor names |
| `attribute` | `[AttributeProto]` | No | Fixed parameters |
| `name` | `string` | Yes | Debug label (not used for wiring) |

### Critical: Input Order is Sacred

Each operator's schema defines the **positional meaning** of inputs. Swapping input order changes the computation:

$$\text{MatMul}([A, B]) = AB \neq BA = \text{MatMul}([B, A])$$

In [ ]:
# Install dependencies (uncomment for Colab)
# !pip install onnx onnxruntime numpy matplotlib

<a id='section-2'></a>
## Section 2: Operator Schemas

### What is a Schema?

Each operator has a **schema** that defines its interface — inputs, outputs, attributes, and type constraints. The schema is versioned (via `since_version`) and defines what the operator computes.

### Schema Components

| Component | Description | Example (MatMul) |
|-----------|-------------|------------------|
| **Inputs** | Positional tensor parameters | `A: T`, `B: T` |
| **Outputs** | Result tensors | `Y: T` |
| **Type constraints** | Polymorphic type variables | `T ∈ {float, double, int32, ...}` |
| **Attributes** | Static configuration | (none for MatMul) |
| **Shape rule** | Output shape computation | `(M,K) × (K,N) → (M,N)` |

### Type Constraints

ONNX uses **type variables** for polymorphism. A constraint like `T ∈ {float16, float32, float64}` means all tensors bound to `T` must share the same type from this set:

$$\text{Add}(A: T, B: T) \to T \quad \text{where } T \in \{\text{float16}, \text{float32}, \text{float64}, \ldots\}$$

This is **parametric polymorphism** — the same operator works with different types but all occurrences of `T` must bind to the same concrete type.

In [ ]:
import numpy as np
from onnx import TensorProto, defs
from onnx.helper import (
    make_model, make_node, make_graph,
    make_tensor_value_info, make_opsetid)
from onnx.numpy_helper import from_array
from onnx.checker import check_model

# Inspect operator schemas
def inspect_schema(op_name, domain=''):
    schema = defs.get_schema(op_name, domain=domain)
    print(f'Operator: {op_name} (since v{schema.since_version})')
    print(f'  Inputs:')
    for inp in schema.inputs:
        print(f'    {inp.name} ({inp.option}): {inp.description[:80]}')
    print(f'  Outputs:')
    for out in schema.outputs:
        print(f'    {out.name}: {out.description[:80]}')
    if schema.attributes:
        print(f'  Attributes:')
        for name, attr in schema.attributes.items():
            print(f'    {name}: type={attr.type}, required={attr.required}')
    print(f'  Type constraints:')
    for tc in schema.type_constraints:
        types = [str(t) for t in tc.allowed_type_strs][:5]
        print(f'    {tc.type_param_str}: {types}...')
    print()

inspect_schema('MatMul')
inspect_schema('Add')
inspect_schema('Transpose')

<a id='section-3'></a>
## Section 3: Edges as Data Flow

### Implicit Edge Construction

ONNX does **not** store explicit edge objects. Instead, edges are inferred from name matching:

$$\text{edge}(v_i, v_j) \iff \exists\, t: t \in \text{outputs}(v_i) \wedge t \in \text{inputs}(v_j)$$

### Edge Properties

| Property | Description |
|----------|------------|
| **Typed** | Each edge carries a tensor with a specific element type |
| **Named** | The tensor name serves as the edge identifier |
| **Unique producer** | Each name is produced exactly once (SSA) |
| **Multiple consumers** | A name can be consumed by many nodes (fan-out) |
| **No fan-in** | A name cannot be produced by multiple sources |

<a id='section-4'></a>
## Section 4: Tensor Types and Element Types

### ONNX Element Type Catalog

| Code | Name | Size | Usage |
|:----:|------|:----:|-------|
| 1 | `FLOAT` | 32-bit | Default for most models |
| 2 | `UINT8` | 8-bit | Quantized inference |
| 3 | `INT8` | 8-bit | Quantized inference |
| 5 | `INT16` | 16-bit | Rare |
| 6 | `INT32` | 32-bit | Indices, shapes |
| 7 | `INT64` | 64-bit | Large-range integers |
| 9 | `BOOL` | 8-bit | Masks, conditions |
| 10 | `FLOAT16` | 16-bit | Half precision (GPU) |
| 11 | `DOUBLE` | 64-bit | High precision |
| 16 | `BFLOAT16` | 16-bit | TPU/some GPUs |

### Tensor Definition

Formally, a tensor $T$ in ONNX is:

$$T \in \mathbb{T}^{d_1 \times d_2 \times \cdots \times d_r}$$

where $\mathbb{T}$ is the element type domain and $r$ is the **rank** (number of dimensions).

### Memory Layout

ONNX tensors use **row-major** (C-style) memory layout:

$$\text{offset}(i_1, i_2, \ldots, i_r) = \sum_{k=1}^{r} i_k \cdot \prod_{j=k+1}^{r} d_j$$

In [ ]:
# Display all ONNX data types
print('ONNX Tensor Data Types')
print('=' * 50)
print(f'{"Code":>5s} | {"Name":>12s} | {"NumPy Equivalent"}')
print('-' * 50)

np_map = {
    1: 'float32', 2: 'uint8', 3: 'int8', 5: 'int16',
    6: 'int32', 7: 'int64', 9: 'bool', 10: 'float16',
    11: 'float64', 12: 'uint32', 13: 'uint64', 16: 'bfloat16'
}

for code, name in sorted(TensorProto.DataType.items(), key=lambda kv: int(kv[1])):
    if int(code) > 0 and int(code) < 20:
        np_equiv = np_map.get(int(code), 'N/A')
        print(f'{int(code):>5d} | {name:>12s} | {np_equiv}')

<a id='section-5'></a>
## Section 5: Tensor Algebra in ONNX

### Shape Inference Rules for Common Operators

| Operator | Input Shapes | Output Shape | Rule |
|:---------|:-------------|:-------------|:-----|
| `MatMul` | $(m,k), (k,n)$ | $(m,n)$ | Matrix multiplication |
| `Add` | $(m,n), (n,)$ | $(m,n)$ | Broadcasting |
| `Relu` | $(d_1, \ldots, d_r)$ | $(d_1, \ldots, d_r)$ | Identity shape |
| `Transpose` | $(d_1, \ldots, d_r)$ | $(d_{\pi(1)}, \ldots, d_{\pi(r)})$ | Permute by `perm` |
| `Reshape` | $(d_1, \ldots, d_r)$ | $(d'_1, \ldots, d'_s)$ | Product preserved |
| `Concat` | $\{(d_1, \ldots, d_i, \ldots)\}_k$ | $(d_1, \ldots, \sum d_i, \ldots)$ | Sum along axis |

### MatMul Shape Rule (Detailed)

For $A \in \mathbb{R}^{\ldots \times m \times k}$ and $B \in \mathbb{R}^{\ldots \times k \times n}$:

$$\text{MatMul}(A, B) \in \mathbb{R}^{\text{broadcast}(\text{batch}) \times m \times n}$$

where batch dimensions follow NumPy broadcasting rules.

<a id='section-6'></a>
## Section 6: Attributes vs Inputs — The Static/Dynamic Boundary

### The Fundamental Distinction

| | **Inputs** | **Attributes** |
|:---|:---:|:---:|
| **Nature** | Tensor data flowing through edges | Fixed parameters inside nodes |
| **When set** | Runtime (changes per inference) | Graph construction (never changes) |
| **Stored in** | Edges (SSA names) | `NodeProto.attribute` |
| **Type** | `TensorProto` | `AttributeProto` (int, float, string, tensor, graph) |
| **Examples** | $X$, $W$ | `perm=[1,0]`, `axis=1`, `alpha=0.01` |

### Why This Matters

Attributes enable **compile-time optimization**: since their values are known before execution, the runtime can:
- Pre-compute output shapes
- Select specialized kernels
- Fuse operators based on attribute values

<a id='section-7'></a>
## Section 7: Building and Inspecting Nodes

In [ ]:
from onnx.numpy_helper import from_array
import onnxruntime as ort

W = from_array(
    np.array([[0.5, -0.3], [0.2, 0.8]], dtype=np.float32), name='W')
b = from_array(
    np.array([0.1, -0.1], dtype=np.float32), name='b')

X = make_tensor_value_info('X', TensorProto.FLOAT, ['batch', 2])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, ['batch', 2])

nodes = [
    make_node('MatMul', ['X', 'W'], ['XW'], name='matmul'),
    make_node('Add', ['XW', 'b'], ['pre_act'], name='add_bias'),
    make_node('Relu', ['pre_act'], ['Y'], name='relu'),
]

graph = make_graph(nodes, 'inspect_demo', [X], [Y], [W, b])
model = make_model(graph, opset_imports=[make_opsetid('', 18)])
check_model(model)

print('Node Inspection Report')
print('=' * 60)
for i, n in enumerate(model.graph.node):
    print(f'\n  Node [{i}]: {n.name}')
    print(f'    op_type:  {n.op_type}')
    print(f'    domain:   {n.domain or "ai.onnx (default)"}')
    print(f'    inputs:   {list(n.input)}')
    print(f'    outputs:  {list(n.output)}')
    if n.attribute:
        for a in n.attribute:
            print(f'    attr:     {a.name} = {list(a.ints) or a.f or a.s}')

print(f'\nInitializers:')
for init in model.graph.initializer:
    dt = TensorProto.DataType.Name(init.data_type)
    print(f'  {init.name}: dtype={dt}, shape={list(init.dims)}')

# Run inference
sess = ort.InferenceSession(model.SerializeToString(),
                            providers=['CPUExecutionProvider'])
x = np.array([[1, 2], [3, 4]], dtype=np.float32)
result = sess.run(None, {'X': x})[0]
print(f'\nResult: {result}')

<a id='section-8'></a>
## Section 8: Key Takeaways & Interview Questions

### Summary

| Concept | Key Point |
|---------|----------|
| **NodeProto** | Single operator invocation with op_type, inputs, outputs, and attributes |
| **Input order** | Positional — defined by the operator schema |
| **Edges** | Implicit via SSA name matching; no explicit edge objects |
| **Type constraints** | Parametric polymorphism via type variables (e.g., `T`) |
| **Element types** | 12+ types from FLOAT to BFLOAT16 |
| **Attributes** | Static (compile-time); Inputs are dynamic (runtime) |

### Interview Questions

1. **Q**: What happens if you swap the input order of a MatMul node?
   - **A**: You get a different computation: `MatMul([A, B])` computes $AB$ while `MatMul([B, A])` computes $BA$. Input order is positional and defined by the operator schema.

2. **Q**: How does ONNX achieve operator polymorphism?
   - **A**: Through type constraints with type variables. `Add(A: T, B: T) → T` means both inputs and the output must share the same concrete type from an allowed set (e.g., float32, float64).

3. **Q**: What is the difference between an input and an attribute on a node?
   - **A**: Inputs are tensor data that flows through edges and can change at runtime. Attributes are fixed parameters embedded in the node that are set at graph construction time and never change (e.g., `perm=[1,0]` in Transpose).

---

**Next:** [ONNX IR Specification](../03_ONNX_IR_Specification/) — The complete protobuf hierarchy.